# Exploratory Data Analysis of the Albanian Telecommunications Sector (2020–2024)

**Master's Thesis:**  
*A Data-Driven Analysis of the Albanian Telecommunications Sector: Trends, Performance, and Forecasting Using National ICT Indicators (2020–2024)*


### Objectives

This notebook performs the exploratory and descriptive analysis of the Albanian telecommunications sector dataset for 2020–2024.

The analysis includes:

1. Data loading and validation
2. Descriptive statistics
3. Absolute and percentage changes
4. Compound annual growth rates (CAGR)
5. Year-on-year (YoY) growth rates
6. Normalized trend indices (2020 = 100)
7. Structural changes in broadband technologies and speed categories
8. Visualization of major telecommunications indicators
9. Identification of the main empirical findings

The analysis uses the cleaned master dataset prepared during the data preparation stage. Missing observations are retained as missing values and are not imputed.

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Figure settings
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import openpyxl

print("openpyxl version:", openpyxl.__version__)
print("openpyxl is working correctly.")

openpyxl version: 3.1.5
openpyxl is working correctly.


In [3]:
from pathlib import Path

print("Python is currently working from:")
print(Path.cwd())

Python is currently working from:
C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\03_scripts\National_Analysis


In [4]:
# ============================================================
# 2. PROJECT PATHS
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\user\Desktop\Albania_Telecom_Thesis_Research"
)

DATA_FILE = (
    PROJECT_ROOT / "02_data" / "National_ICT"
    / "Albania_Telecom_Master_2020_2024_V1_FINAL.xlsx"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "04_outputs" / "Reproduced_Results"
    / "National_Analysis"
)

FIGURE_DIR = OUTPUT_ROOT / "Figures"
TABLE_DIR = OUTPUT_ROOT / "Descriptive_Tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Data file:", DATA_FILE)
print("Data file exists:", DATA_FILE.exists())
print("Figures folder:", FIGURE_DIR.resolve())
print("Tables folder:", TABLE_DIR.resolve())

Data file: C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\02_data\National_ICT\Albania_Telecom_Master_2020_2024_V1_FINAL.xlsx
Data file exists: True
Figures folder: C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\04_outputs\Reproduced_Results\National_Analysis\Figures
Tables folder: C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\04_outputs\Reproduced_Results\National_Analysis\Descriptive_Tables


In [5]:
# ============================================================
# 3. LOAD DATA
# ============================================================

master = pd.read_excel(DATA_FILE, sheet_name="Master_Data")
dictionary = pd.read_excel(DATA_FILE, sheet_name="Data_Dictionary")
kpi_selection = pd.read_excel(DATA_FILE, sheet_name="KPI_Selection")
missing_data = pd.read_excel(DATA_FILE, sheet_name="Missing_Data")

print("Workbook loaded successfully.")
print()

print(f"Master_Data:     {master.shape[0]} rows × {master.shape[1]} columns")
print(f"Data_Dictionary: {dictionary.shape[0]} rows × {dictionary.shape[1]} columns")
print(f"KPI_Selection:   {kpi_selection.shape[0]} rows × {kpi_selection.shape[1]} columns")
print(f"Missing_Data:    {missing_data.shape[0]} rows × {missing_data.shape[1]} columns")

Workbook loaded successfully.

Master_Data:     410 rows × 6 columns
Data_Dictionary: 82 rows × 15 columns
KPI_Selection:   20 rows × 6 columns
Missing_Data:    97 rows × 5 columns


In [6]:
master.head(10)

,IndicatorCode,Indicator,Category,Unit,Year,Value
0,i112,Fixed-telephone subscriptions,Fixed Telephone Network,Subscriptions,2024,"165,380.00"
1,i116,Fixed-telephone residential subscriptions,Fixed Telephone Network,Subscriptions,2024,79.00
2,i112IP,VoIP subscriptions,Fixed Telephone Network,Subscriptions,2024,"18,078.00"
3,i112pt,Fixed-telephone numbers ported,Fixed Telephone Network,Numbers,2024,173.00
4,i271,Mobile-cellular telephone subscriptions,Mobile Cellular Network,Subscriptions,2024,"2,488,380.00"
5,i271p,Prepaid mobile-cellular telephone subscriptions,Mobile Cellular Network,Subscriptions,2024,"1,431,673.00"
6,i271m2m,M2M mobile-network subscriptions,Mobile Cellular Network,Subscriptions,2024,"151,642.00"
7,i271pt,Mobile-cellular numbers ported,Mobile Cellular Network,Numbers,2024,"19,452.00"
8,i271_spec_a,"Amount of spectrum offered for IMT systems, in...",Mobile Cellular Network,MHz,2024,"4,020.00"
9,i271_spec_a1,Amount of spectrum offered for IMT systems in ...,Mobile Cellular Network,MHz,2024,95.00


## 1. Data Loading and Validation

The cleaned master telecommunications dataset is imported from the final data preparation workbook. Supporting sheets containing the data dictionary, KPI selection, and missing-data documentation are also loaded to ensure consistency with the research design established during the data preparation stage.

### 1.1 Structural Validation

Before performing the exploratory analysis, the dataset is checked for its dimensions, variables, temporal coverage, unique indicators, missing observations, duplicate indicator-year records, and data types.

In [7]:
# ============================================================
# 4. STRUCTURAL VALIDATION
# ============================================================

print("=" * 60)
print("MASTER DATA VALIDATION")
print("=" * 60)

print("\nDataset dimensions:")
print(f"{master.shape[0]} rows × {master.shape[1]} columns")

print("\nColumns:")
print(master.columns.tolist())

print("\nYear coverage:")
print(sorted(master["Year"].dropna().unique()))

print("\nNumber of unique indicators:")
print(master["IndicatorCode"].nunique())

print("\nData types:")
print(master.dtypes)

print("\nMissing values by column:")
print(master.isna().sum())

duplicate_count = master.duplicated(
    subset=["IndicatorCode", "Year"]
).sum()

print("\nDuplicate IndicatorCode-Year observations:")
print(duplicate_count)

MASTER DATA VALIDATION

Dataset dimensions:
410 rows × 6 columns

Columns:
['IndicatorCode', 'Indicator', 'Category', 'Unit', 'Year', 'Value']

Year coverage:
[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Number of unique indicators:
82

Data types:
IndicatorCode     object
Indicator         object
Category          object
Unit              object
Year               int64
Value            float64
dtype: object

Missing values by column:
IndicatorCode     0
Indicator         0
Category          0
Unit              0
Year              0
Value            96
dtype: int64

Duplicate IndicatorCode-Year observations:
0


In [8]:
# ============================================================
# 5. AUTOMATED VALIDATION CHECKS
# ============================================================

expected_columns = {
    "IndicatorCode",
    "Indicator",
    "Category",
    "Unit",
    "Year",
    "Value"
}

expected_years = {2020, 2021, 2022, 2023, 2024}

actual_columns = set(master.columns)
actual_years = set(master["Year"].dropna().astype(int).unique())

assert expected_columns.issubset(actual_columns), \
    "ERROR: One or more required columns are missing."

assert actual_years == expected_years, \
    f"ERROR: Unexpected year coverage: {sorted(actual_years)}"

assert master.duplicated(
    subset=["IndicatorCode", "Year"]
).sum() == 0, \
    "ERROR: Duplicate indicator-year observations detected."

print("✓ Required columns verified")
print("✓ Year coverage verified: 2020–2024")
print("✓ No duplicate indicator-year observations detected")
print()
print("Basic structural validation PASSED.")

✓ Required columns verified
✓ Year coverage verified: 2020–2024
✓ No duplicate indicator-year observations detected

Basic structural validation PASSED.


In [9]:
# ============================================================
# 6. NUMERIC VALIDATION
# ============================================================

master["Value"] = pd.to_numeric(
    master["Value"],
    errors="coerce"
)

print("Value data type:", master["Value"].dtype)
print("Missing Value observations:", master["Value"].isna().sum())

Value data type: float64
Missing Value observations: 96


### 1.2 KPI Selection Validation

The KPI selection established during the research-design stage is validated against the master dataset. Each selected KPI code must correspond to an indicator available in the master dataset before exploratory analysis is performed.

In [10]:
# ============================================================
# 7. KPI SELECTION VALIDATION
# ============================================================

selected_codes = (
    kpi_selection["Code"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

master_codes = set(
    master["IndicatorCode"]
    .dropna()
    .astype(str)
    .str.strip()
)

matched_codes = [
    code for code in selected_codes
    if code in master_codes
]

unmatched_codes = [
    code for code in selected_codes
    if code not in master_codes
]

print("Number of selected KPIs:", len(selected_codes))
print("Number matched to Master_Data:", len(matched_codes))
print("Number unmatched:", len(unmatched_codes))

if unmatched_codes:
    print("\nWARNING — unmatched KPI codes:")
    for code in unmatched_codes:
        print(" -", code)
else:
    print("\n✓ All selected KPI codes match Master_Data.")

Number of selected KPIs: 20
Number matched to Master_Data: 20
Number unmatched: 0

✓ All selected KPI codes match Master_Data.


## 2. Preparation of the Analytical KPI Dataset

The exploratory analysis focuses on the 20 KPIs selected during the research-design stage. These indicators represent the principal dimensions of sector development, including broadband adoption, technological transformation, network usage, traffic, revenue, investment, and service evolution.

Before calculating growth measures, the temporal coverage of each selected KPI is examined. Indicators with incomplete historical coverage are retained in the analysis, but missing observations are not imputed.

In [11]:
# ============================================================
# 8. CREATE KPI ANALYSIS DATASET
# ============================================================

kpi_data = (
    master[
        master["IndicatorCode"].isin(selected_codes)
    ]
    .copy()
    .sort_values(["IndicatorCode", "Year"])
    .reset_index(drop=True)
)

print("=" * 60)
print("KPI ANALYSIS DATASET")
print("=" * 60)

print("\nRows:", len(kpi_data))
print("Unique KPIs:", kpi_data["IndicatorCode"].nunique())
print("Years:", sorted(kpi_data["Year"].unique()))
print("Missing values:", kpi_data["Value"].isna().sum())

kpi_data.head(10)

KPI ANALYSIS DATASET

Rows: 100
Unique KPIs: 20
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Missing values: 12


,IndicatorCode,Indicator,Category,Unit,Year,Value
0,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2020,"82,641,199.00"
1,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2021,"84,784,753.00"
2,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2022,"42,610,420.00"
3,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2023,"29,683,565.00"
4,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2024,"26,557,212.00"
5,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2020,NaN
6,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2021,NaN
7,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2022,"5,043,679,027.00"
8,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2023,"4,319,090,725.00"
9,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2024,"4,108,613,047.00"


### 2.1 Temporal Coverage

Temporal coverage is examined before growth calculations are performed. For each KPI, the number of valid and missing observations and the first and last available years are identified.

This distinction is important because some indicators were not reported throughout the entire 2020–2024 period. Growth calculations for such indicators must therefore reflect their actual period of data availability rather than assuming observations that do not exist.

In [12]:
# ============================================================
# 9. KPI TEMPORAL COVERAGE
# ============================================================

coverage_records = []

for code, group in kpi_data.groupby("IndicatorCode"):

    group = group.sort_values("Year")
    available = group.dropna(subset=["Value"])

    coverage_records.append({
        "IndicatorCode": code,
        "Indicator": group["Indicator"].iloc[0],
        "Unit": group["Unit"].iloc[0],
        "Available_Observations": len(available),
        "Missing_Observations": group["Value"].isna().sum(),
        "First_Available_Year":
            int(available["Year"].min())
            if not available.empty else np.nan,
        "Last_Available_Year":
            int(available["Year"].max())
            if not available.empty else np.nan
    })

coverage = pd.DataFrame(coverage_records)

coverage

,IndicatorCode,Indicator,Unit,Available_Observations,Missing_Observations,First_Available_Year,Last_Available_Year
0,i133fo,Total outgoing voice traffic from (originated ...,minutes,5,0,2020,2024
1,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,3,2,2022,2024
2,i133sms,"SMS sent,",Messages,5,0,2020,2024
3,i135tfb,Fixed- broadband Internet traffic (exabytes),Exabytes,5,0,2020,2024
4,i136mwi,Mobile-broadband Internet traffic (within the ...,Exabytes,5,0,2020,2024
5,i271G5_pop,At least 5G network,%,4,1,2021,2024
6,i271GA,At least LTE/WiMAX mobile network,%,5,0,2020,2024
7,i271fw,Terrestrial fixed wireless broadband subscript...,Subscriptions,3,2,2022,2024
8,i271mw,Number of active mobile-broadband subscriptions,Subscriptions,5,0,2020,2024
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,Subscriptions,3,2,2022,2024


In [13]:
incomplete_kpis = coverage[
    coverage["Available_Observations"] < 5
].copy()

print("KPIs without complete 2020–2024 coverage:")
print()

incomplete_kpis

KPIs without complete 2020–2024 coverage:



,IndicatorCode,Indicator,Unit,Available_Observations,Missing_Observations,First_Available_Year,Last_Available_Year
1,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,3,2,2022,2024
5,i271G5_pop,At least 5G network,%,4,1,2021,2024
7,i271fw,Terrestrial fixed wireless broadband subscript...,Subscriptions,3,2,2022,2024
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,Subscriptions,3,2,2022,2024
13,i4213dsl,DSL Internet subscriptions DSL,Subscriptions,3,2,2022,2024
14,i4213ftth/b,Fibre-to-the-home/building Internet subscriptions,Subscriptions,3,2,2022,2024
16,i51,Persons employed in full-time equivalents,People,4,1,2020,2024


In [14]:
# ============================================================
# 10. SAVE TEMPORAL COVERAGE TABLE
# ============================================================



coverage.to_excel(
    TABLE_DIR / "kpi_temporal_coverage.xlsx",
    index=False
)

print("✓ Saved: outputs/kpi_temporal_coverage.xlsx")

✓ Saved: outputs/kpi_temporal_coverage.xlsx


## 3. Descriptive Statistics

Descriptive statistics are calculated for each selected KPI using all valid observations available during the 2020–2024 study period.

For each indicator, the number of observations, mean, median, standard deviation, minimum, and maximum are reported. Because some indicators have incomplete temporal coverage, the number of valid observations is explicitly retained. Missing observations are excluded from the calculations and are not imputed.

In [15]:
# ============================================================
# 11. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_stats = (
    kpi_data
    .groupby(
        ["IndicatorCode", "Indicator", "Category", "Unit"]
    )["Value"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Std_Dev="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
)

print("Number of KPIs summarized:", len(descriptive_stats))

descriptive_stats

Number of KPIs summarized: 20


,IndicatorCode,Indicator,Category,Unit,Count,Mean,Median,Std_Dev,Minimum,Maximum
0,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,5,"53,255,429.80","42,610,420.00","28,457,712.18","26,557,212.00","84,784,753.00"
1,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,3,"4,490,460,933.00","4,319,090,725.00","490,523,096.17","4,108,613,047.00","5,043,679,027.00"
2,i133sms,"SMS sent,",Traffic,Messages,5,"513,440,063.75","553,607,871.00","113,799,674.53","384,719,599.00","660,604,925.00"
3,i135tfb,Fixed- broadband Internet traffic (exabytes),Traffic,Exabytes,5,1.12,1.02,0.44,0.61,1.78
4,i136mwi,Mobile-broadband Internet traffic (within the ...,Traffic,Exabytes,5,0.17,0.18,0.05,0.09,0.22
5,i271G5_pop,At least 5G network,Mobile Network,%,4,6.00,0.00,12.00,0.00,24.00
6,i271GA,At least LTE/WiMAX mobile network,Mobile Network,%,5,99.10,99.00,0.51,98.40,99.60
7,i271fw,Terrestrial fixed wireless broadband subscript...,Fixed Broadband,Subscriptions,3,"30,420.33","30,968.00","21,628.70","8,523.00","51,770.00"
8,i271mw,Number of active mobile-broadband subscriptions,Mobile Broadband,Subscriptions,5,"2,073,798.80","2,082,525.00","51,833.68","1,998,579.00","2,140,180.00"
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,Mobile Broadband,Subscriptions,3,"2,502,433.67","2,593,165.00","174,485.75","2,301,276.00","2,612,860.00"


In [16]:
descriptive_stats_export = descriptive_stats.copy()

columns_to_round = [
    "Mean",
    "Median",
    "Std_Dev",
    "Minimum",
    "Maximum"
]

descriptive_stats_export[columns_to_round] = (
    descriptive_stats_export[columns_to_round].round(2)
)

descriptive_stats_export

,IndicatorCode,Indicator,Category,Unit,Count,Mean,Median,Std_Dev,Minimum,Maximum
0,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,5,"53,255,429.80","42,610,420.00","28,457,712.18","26,557,212.00","84,784,753.00"
1,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,3,"4,490,460,933.00","4,319,090,725.00","490,523,096.17","4,108,613,047.00","5,043,679,027.00"
2,i133sms,"SMS sent,",Traffic,Messages,5,"513,440,063.75","553,607,871.00","113,799,674.53","384,719,599.00","660,604,925.00"
3,i135tfb,Fixed- broadband Internet traffic (exabytes),Traffic,Exabytes,5,1.12,1.02,0.44,0.61,1.78
4,i136mwi,Mobile-broadband Internet traffic (within the ...,Traffic,Exabytes,5,0.17,0.18,0.05,0.09,0.22
5,i271G5_pop,At least 5G network,Mobile Network,%,4,6.00,0.00,12.00,0.00,24.00
6,i271GA,At least LTE/WiMAX mobile network,Mobile Network,%,5,99.10,99.00,0.51,98.40,99.60
7,i271fw,Terrestrial fixed wireless broadband subscript...,Fixed Broadband,Subscriptions,3,"30,420.33","30,968.00","21,628.70","8,523.00","51,770.00"
8,i271mw,Number of active mobile-broadband subscriptions,Mobile Broadband,Subscriptions,5,"2,073,798.80","2,082,525.00","51,833.68","1,998,579.00","2,140,180.00"
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,Mobile Broadband,Subscriptions,3,"2,502,433.67","2,593,165.00","174,485.75","2,301,276.00","2,612,860.00"


In [17]:
descriptive_stats_export.to_excel(
    TABLE_DIR / "descriptive_statistics.xlsx",
    index=False
)

print("✓ Saved: outputs/descriptive_statistics.xlsx")

✓ Saved: outputs/descriptive_statistics.xlsx


## 4. KPI Growth Summary

Long-term changes in the selected KPIs are evaluated using absolute change, percentage change, and compound annual growth rate (CAGR).

Absolute change is defined as:

\[
\Delta X = X_{end} - X_{start}
\]

Percentage change is defined as:

\[
\%\Delta = \frac{X_{end} - X_{start}}{X_{start}} \times 100
\]

CAGR is defined as:

\[
CAGR = \left(\frac{X_{end}}{X_{start}}\right)^{1/n} - 1
\]

where \(n\) represents the number of years between the first and last valid observations.

For indicators with complete temporal coverage, the comparison period is 2020–2024 and \(n=4\). For indicators without a valid 2020 observation, calculations use the first and last available observations and the corresponding period is explicitly reported. No missing values are imputed.

In [18]:
# ============================================================
# 12. KPI GROWTH SUMMARY
# ============================================================

summary_records = []

for code, group in kpi_data.groupby("IndicatorCode"):

    group = group.sort_values("Year")
    valid = group.dropna(subset=["Value"])

    if valid.empty:
        continue

    start = valid.iloc[0]
    end = valid.iloc[-1]

    start_year = int(start["Year"])
    end_year = int(end["Year"])

    start_value = float(start["Value"])
    end_value = float(end["Value"])

    # Absolute change
    absolute_change = end_value - start_value

    # Percentage change
    if start_value != 0:
        percentage_change = (
            (end_value - start_value) / start_value
        ) * 100
    else:
        percentage_change = np.nan

    # Number of years in actual comparison period
    number_of_years = end_year - start_year

    # CAGR
    if (
        number_of_years > 0
        and start_value > 0
        and end_value > 0
    ):
        cagr = (
            (end_value / start_value)
            ** (1 / number_of_years)
            - 1
        ) * 100
    else:
        cagr = np.nan

    summary_records.append({
        "IndicatorCode": code,
        "Indicator": group["Indicator"].iloc[0],
        "Category": group["Category"].iloc[0],
        "Unit": group["Unit"].iloc[0],
        "Start_Year": start_year,
        "End_Year": end_year,
        "Years": number_of_years,
        "Start_Value": start_value,
        "End_Value": end_value,
        "Absolute_Change": absolute_change,
        "Percentage_Change": percentage_change,
        "CAGR_Percent": cagr
    })

kpi_summary = pd.DataFrame(summary_records)

print("Number of KPIs in growth summary:", len(kpi_summary))

kpi_summary

Number of KPIs in growth summary: 20


,IndicatorCode,Indicator,Category,Unit,Start_Year,End_Year,Years,Start_Value,End_Value,Absolute_Change,Percentage_Change,CAGR_Percent
0,i133fo,Total outgoing voice traffic from (originated ...,Traffic,minutes,2020,2024,4,"82,641,199.00","26,557,212.00","-56,083,987.00",-67.86,-24.71
1,i133mm,"Total Outgoing Mobile traffic, in minutes",Traffic,minutes,2022,2024,2,"5,043,679,027.00","4,108,613,047.00","-935,065,980.00",-18.54,-9.74
2,i133sms,"SMS sent,",Traffic,Messages,2020,2024,4,"660,604,925.00","384,719,599.00","-275,885,326.00",-41.76,-12.64
3,i135tfb,Fixed- broadband Internet traffic (exabytes),Traffic,Exabytes,2020,2024,4,0.61,1.78,1.17,192.58,30.79
4,i136mwi,Mobile-broadband Internet traffic (within the ...,Traffic,Exabytes,2020,2024,4,0.09,0.22,0.13,151.19,25.89
5,i271G5_pop,At least 5G network,Mobile Network,%,2021,2024,3,0.00,24.00,24.00,NaN,NaN
6,i271GA,At least LTE/WiMAX mobile network,Mobile Network,%,2020,2024,4,98.40,99.60,1.20,1.22,0.30
7,i271fw,Terrestrial fixed wireless broadband subscript...,Fixed Broadband,Subscriptions,2022,2024,2,"8,523.00","51,770.00","43,247.00",507.42,146.46
8,i271mw,Number of active mobile-broadband subscriptions,Mobile Broadband,Subscriptions,2020,2024,4,"1,998,579.00","2,091,266.00","92,687.00",4.64,1.14
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,Mobile Broadband,Subscriptions,2022,2024,2,"2,301,276.00","2,612,860.00","311,584.00",13.54,6.55


In [19]:
kpi_summary["Trend_Direction"] = np.select(
    [
        kpi_summary["Absolute_Change"] > 0,
        kpi_summary["Absolute_Change"] < 0,
        kpi_summary["Absolute_Change"] == 0
    ],
    [
        "Increase",
        "Decrease",
        "No change"
    ],
    default="Insufficient data"
)

kpi_summary[
    [
        "IndicatorCode",
        "Indicator",
        "Start_Year",
        "End_Year",
        "Percentage_Change",
        "CAGR_Percent",
        "Trend_Direction"
    ]
]

,IndicatorCode,Indicator,Start_Year,End_Year,Percentage_Change,CAGR_Percent,Trend_Direction
0,i133fo,Total outgoing voice traffic from (originated ...,2020,2024,-67.86,-24.71,Decrease
1,i133mm,"Total Outgoing Mobile traffic, in minutes",2022,2024,-18.54,-9.74,Decrease
2,i133sms,"SMS sent,",2020,2024,-41.76,-12.64,Decrease
3,i135tfb,Fixed- broadband Internet traffic (exabytes),2020,2024,192.58,30.79,Increase
4,i136mwi,Mobile-broadband Internet traffic (within the ...,2020,2024,151.19,25.89,Increase
5,i271G5_pop,At least 5G network,2021,2024,NaN,NaN,Increase
6,i271GA,At least LTE/WiMAX mobile network,2020,2024,1.22,0.30,Increase
7,i271fw,Terrestrial fixed wireless broadband subscript...,2022,2024,507.42,146.46,Increase
8,i271mw,Number of active mobile-broadband subscriptions,2020,2024,4.64,1.14,Increase
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,2022,2024,13.54,6.55,Increase


In [20]:
full_period_summary = kpi_summary[
    (kpi_summary["Start_Year"] == 2020)
    & (kpi_summary["End_Year"] == 2024)
].copy()

shorter_period_summary = kpi_summary[
    ~(
        (kpi_summary["Start_Year"] == 2020)
        & (kpi_summary["End_Year"] == 2024)
    )
].copy()

print(
    "KPIs with complete 2020–2024 growth comparison:",
    len(full_period_summary)
)

print(
    "KPIs using a shorter available comparison period:",
    len(shorter_period_summary)
)

shorter_period_summary[
    [
        "IndicatorCode",
        "Indicator",
        "Start_Year",
        "End_Year",
        "Percentage_Change",
        "CAGR_Percent"
    ]
]

KPIs with complete 2020–2024 growth comparison: 14
KPIs using a shorter available comparison period: 6


,IndicatorCode,Indicator,Start_Year,End_Year,Percentage_Change,CAGR_Percent
1,i133mm,"Total Outgoing Mobile traffic, in minutes",2022,2024,-18.54,-9.74
5,i271G5_pop,At least 5G network,2021,2024,NaN,NaN
7,i271fw,Terrestrial fixed wireless broadband subscript...,2022,2024,507.42,146.46
9,i271mwa,Active subscriptions to LTE/WiMAX mobile-broad...,2022,2024,13.54,6.55
13,i4213dsl,DSL Internet subscriptions DSL,2022,2024,-57.95,-35.16
14,i4213ftth/b,Fibre-to-the-home/building Internet subscriptions,2022,2024,35.20,16.27


In [21]:
kpi_summary_export = kpi_summary.copy()

columns_to_round = [
    "Start_Value",
    "End_Value",
    "Absolute_Change",
    "Percentage_Change",
    "CAGR_Percent"
]

kpi_summary_export[columns_to_round] = (
    kpi_summary_export[columns_to_round].round(2)
)

kpi_summary_export.to_excel(
    TABLE_DIR / "kpi_summary.xlsx",
    index=False
)

print("✓ Saved: outputs/kpi_summary.xlsx")

✓ Saved: outputs/kpi_summary.xlsx


## 5. Year-on-Year Growth

Year-on-year (YoY) growth rates are calculated to examine short-term changes in each KPI and identify periods of acceleration or decline.

For year \(t\):

\[
YoY_t = \frac{X_t-X_{t-1}}{X_{t-1}}\times100
\]

A YoY growth rate is calculated only when valid observations exist for both the current and immediately preceding year. The first observation of each series therefore has no YoY growth rate.

In [22]:
# ============================================================
# 13. YEAR-ON-YEAR GROWTH
# ============================================================

kpi_data = kpi_data.sort_values(
    ["IndicatorCode", "Year"]
).copy()

kpi_data["Previous_Value"] = (
    kpi_data
    .groupby("IndicatorCode")["Value"]
    .shift(1)
)

kpi_data["Previous_Year"] = (
    kpi_data
    .groupby("IndicatorCode")["Year"]
    .shift(1)
)

valid_yoy = (
    kpi_data["Previous_Value"].notna()
    & kpi_data["Value"].notna()
    & (kpi_data["Previous_Value"] != 0)
    & ((kpi_data["Year"] - kpi_data["Previous_Year"]) == 1)
)

kpi_data["YoY_Growth_Percent"] = np.where(
    valid_yoy,
    (
        (kpi_data["Value"] - kpi_data["Previous_Value"])
        / kpi_data["Previous_Value"]
    ) * 100,
    np.nan
)

yoy_table = kpi_data[
    [
        "IndicatorCode",
        "Indicator",
        "Unit",
        "Year",
        "Value",
        "Previous_Value",
        "YoY_Growth_Percent"
    ]
].copy()

yoy_table

,IndicatorCode,Indicator,Unit,Year,Value,Previous_Value,YoY_Growth_Percent
0,i133fo,Total outgoing voice traffic from (originated ...,minutes,2020,"82,641,199.00",NaN,NaN
1,i133fo,Total outgoing voice traffic from (originated ...,minutes,2021,"84,784,753.00","82,641,199.00",2.59
2,i133fo,Total outgoing voice traffic from (originated ...,minutes,2022,"42,610,420.00","84,784,753.00",-49.74
3,i133fo,Total outgoing voice traffic from (originated ...,minutes,2023,"29,683,565.00","42,610,420.00",-30.34
4,i133fo,Total outgoing voice traffic from (originated ...,minutes,2024,"26,557,212.00","29,683,565.00",-10.53
5,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2020,NaN,NaN,NaN
6,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2021,NaN,NaN,NaN
7,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2022,"5,043,679,027.00",NaN,NaN
8,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2023,"4,319,090,725.00","5,043,679,027.00",-14.37
9,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2024,"4,108,613,047.00","4,319,090,725.00",-4.87


In [23]:
yoy_export = yoy_table.copy()

yoy_export["YoY_Growth_Percent"] = (
    yoy_export["YoY_Growth_Percent"].round(2)
)

yoy_export.to_excel(
    TABLE_DIR / "yoy_growth.xlsx",
    index=False
)

print("✓ Saved: outputs/yoy_growth.xlsx")

✓ Saved: outputs/yoy_growth.xlsx


## 6. Normalized Trend Analysis

To facilitate comparison between indicators measured in different units, KPIs with valid 2020 observations are transformed into index values using 2020 as the base year.

\[
Index_t = \frac{X_t}{X_{2020}}\times100
\]

An index value of 100 represents the indicator's 2020 level. Values above 100 indicate an increase relative to 2020, while values below 100 indicate a decline.

Indicators without a valid 2020 observation are not assigned a 2020-based index.

In [24]:
# ============================================================
# 14. NORMALIZED TRENDS — 2020 = 100
# ============================================================

base_2020 = (
    kpi_data[kpi_data["Year"] == 2020]
    [["IndicatorCode", "Value"]]
    .rename(columns={"Value": "Value_2020"})
)

normalized = kpi_data.merge(
    base_2020,
    on="IndicatorCode",
    how="left"
)

normalized["Index_2020_100"] = np.where(
    normalized["Value_2020"].notna()
    & (normalized["Value_2020"] != 0),

    (
        normalized["Value"]
        / normalized["Value_2020"]
    ) * 100,

    np.nan
)

normalized_table = normalized[
    [
        "IndicatorCode",
        "Indicator",
        "Unit",
        "Year",
        "Value",
        "Value_2020",
        "Index_2020_100"
    ]
].copy()

normalized_table.head(20)

,IndicatorCode,Indicator,Unit,Year,Value,Value_2020,Index_2020_100
0,i133fo,Total outgoing voice traffic from (originated ...,minutes,2020,"82,641,199.00","82,641,199.00",100.00
1,i133fo,Total outgoing voice traffic from (originated ...,minutes,2021,"84,784,753.00","82,641,199.00",102.59
2,i133fo,Total outgoing voice traffic from (originated ...,minutes,2022,"42,610,420.00","82,641,199.00",51.56
3,i133fo,Total outgoing voice traffic from (originated ...,minutes,2023,"29,683,565.00","82,641,199.00",35.92
4,i133fo,Total outgoing voice traffic from (originated ...,minutes,2024,"26,557,212.00","82,641,199.00",32.14
5,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2020,NaN,NaN,NaN
6,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2021,NaN,NaN,NaN
7,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2022,"5,043,679,027.00",NaN,NaN
8,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2023,"4,319,090,725.00",NaN,NaN
9,i133mm,"Total Outgoing Mobile traffic, in minutes",minutes,2024,"4,108,613,047.00",NaN,NaN


In [25]:
check_2020 = normalized_table[
    (normalized_table["Year"] == 2020)
    & normalized_table["Index_2020_100"].notna()
]

assert np.allclose(
    check_2020["Index_2020_100"],
    100
)

print(
    f"✓ Normalization verified for "
    f"{len(check_2020)} KPIs with valid 2020 observations."
)

✓ Normalization verified for 14 KPIs with valid 2020 observations.


In [26]:
normalized_export = normalized_table.copy()

normalized_export["Index_2020_100"] = (
    normalized_export["Index_2020_100"].round(2)
)

normalized_export.to_excel(
    TABLE_DIR / "normalized_trends_2020_100.xlsx",
    index=False
)

print("✓ Saved: outputs/normalized_trends_2020_100.xlsx")

✓ Saved: outputs/normalized_trends_2020_100.xlsx

In [27]:
# ============================================================
# 15. DAY 2 NUMERICAL OUTPUT CHECK
# ============================================================

output_files = [
    TABLE_DIR / "kpi_temporal_coverage.xlsx",
    TABLE_DIR / "descriptive_statistics.xlsx",
    TABLE_DIR / "kpi_summary.xlsx",
    TABLE_DIR / "yoy_growth.xlsx",
    TABLE_DIR / "normalized_trends_2020_100.xlsx"
]

print("=" * 60)
print("DAY 2 NUMERICAL OUTPUT CHECK")
print("=" * 60)

for file in output_files:
    status = "✓ EXISTS" if file.exists() else "✗ MISSING"
    print(f"{status}: {file.name}")

print()
print("Selected KPIs:", len(selected_codes))
print("KPIs in analytical dataset:", kpi_data["IndicatorCode"].nunique())
print("KPIs in descriptive statistics:", len(descriptive_stats))
print("KPIs in growth summary:", len(kpi_summary))

DAY 2 NUMERICAL OUTPUT CHECK
✓ EXISTS: kpi_temporal_coverage.xlsx
✓ EXISTS: descriptive_statistics.xlsx
✓ EXISTS: kpi_summary.xlsx
✓ EXISTS: yoy_growth.xlsx
✓ EXISTS: normalized_trends_2020_100.xlsx

Selected KPIs: 20
KPIs in analytical dataset: 20
KPIs in descriptive statistics: 20
KPIs in growth summary: 20


## 7. Exploratory Visualization

The following figures visualize the principal trends identified in the descriptive and growth analysis. Figures are selected according to their relevance to the research questions rather than plotting every available indicator individually.

The visual analysis focuses on broadband adoption, technological transformation, network usage, data traffic, legacy communication services, sector revenue, investment, and normalized sector development.

In [28]:
# ============================================================
# 16. FIGURE PREPARATION
# ============================================================



YEARS = [2020, 2021, 2022, 2023, 2024]

def get_indicator(code):
    """Return one indicator series sorted by year."""
    return (
        master[master["IndicatorCode"] == code]
        .sort_values("Year")
        .copy()
    )

def save_figure(filename):
    """Save current matplotlib figure at thesis-quality resolution."""
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

print("✓ Figure directory ready:", FIGURE_DIR.resolve())

✓ Figure directory ready: C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\04_outputs\Reproduced_Results\National_Analysis\Figures


### Figure 1. Fixed Broadband Subscriptions

The first figure examines the overall development of fixed broadband subscriptions during 2020–2024.

In [29]:
# ============================================================
# FIGURE 1 — FIXED BROADBAND SUBSCRIPTIONS
# ============================================================

data = get_indicator("i4213tfbbb")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    data["Year"],
    data["Value"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Fixed Broadband Subscriptions in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Subscriptions")

ax.set_xticks(YEARS)

ax.ticklabel_format(
    style="plain",
    axis="y"
)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_01_Fixed_Broadband_Subscriptions.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 2. Fixed Broadband Technology Transformation

FTTH/B and DSL subscriptions are compared over their common period of data availability (2022–2024) to examine the technological transformation of Albania's fixed broadband infrastructure.

In [30]:
# ============================================================
# FIGURE 2 — FTTH/B VS DSL
# ============================================================

ftth = get_indicator("i4213ftth/b")
dsl = get_indicator("i4213dsl")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    ftth["Year"],
    ftth["Value"],
    marker="o",
    linewidth=2,
    label="FTTH/B"
)

ax.plot(
    dsl["Year"],
    dsl["Value"],
    marker="o",
    linewidth=2,
    label="DSL"
)

ax.set_title(
    "FTTH/B and DSL Subscriptions in Albania, 2022–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Subscriptions")

ax.set_xticks([2022, 2023, 2024])

ax.ticklabel_format(
    style="plain",
    axis="y"
)

ax.legend(frameon=False)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_02_FTTH_vs_DSL.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 3. Fixed Broadband Speed Transformation

Fixed broadband subscriptions are compared across three speed categories to examine the migration from lower-speed connections toward higher-speed broadband services.

In [31]:
# ============================================================
# FIGURE 3 — BROADBAND SPEED CATEGORIES
# ============================================================

speed_10_30 = get_indicator("i4213_10to30")
speed_30_100 = get_indicator("i4213_30to100")
speed_100_plus = get_indicator("i4213_G100")

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    speed_10_30["Year"],
    speed_10_30["Value"],
    marker="o",
    linewidth=2,
    label="10–30 Mbit/s"
)

ax.plot(
    speed_30_100["Year"],
    speed_30_100["Value"],
    marker="o",
    linewidth=2,
    label="30–100 Mbit/s"
)

ax.plot(
    speed_100_plus["Year"],
    speed_100_plus["Value"],
    marker="o",
    linewidth=2,
    label="≥100 Mbit/s"
)

ax.set_title(
    "Fixed Broadband Subscriptions by Speed Category, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Subscriptions")

ax.set_xticks(YEARS)

ax.ticklabel_format(
    style="plain",
    axis="y"
)

ax.legend(frameon=False)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_03_Broadband_Speed_Transformation.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 4. Terrestrial Fixed Wireless Broadband

Terrestrial fixed wireless broadband subscriptions are examined over their available 2022–2024 reporting period.

In [32]:
# ============================================================
# FIGURE 4 — FIXED WIRELESS BROADBAND
# ============================================================

data = get_indicator("i271fw")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    data["Year"],
    data["Value"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Terrestrial Fixed Wireless Broadband Subscriptions, 2022–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Subscriptions")

ax.set_xticks([2022, 2023, 2024])

ax.ticklabel_format(
    style="plain",
    axis="y"
)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_04_Fixed_Wireless_Broadband.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 5. Active Mobile Broadband Subscriptions

The development of active mobile broadband subscriptions is examined over the full 2020–2024 study period.

In [33]:
# ============================================================
# FIGURE 5 — MOBILE BROADBAND
# ============================================================

data = get_indicator("i271mw")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    data["Year"],
    data["Value"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Active Mobile Broadband Subscriptions, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Subscriptions")
ax.set_xticks(YEARS)

ax.ticklabel_format(
    style="plain",
    axis="y"
)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_05_Mobile_Broadband_Subscriptions.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 6. Mobile Network Coverage

LTE/WiMAX and 5G population coverage are compared to illustrate the evolution from mature 4G coverage toward the emergence of 5G infrastructure.

In [34]:
# ============================================================
# FIGURE 6 — LTE AND 5G COVERAGE
# ============================================================

lte = get_indicator("i271GA")
g5 = get_indicator("i271G5_pop")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    lte["Year"],
    lte["Value"],
    marker="o",
    linewidth=2,
    label="LTE/WiMAX"
)

ax.plot(
    g5["Year"],
    g5["Value"],
    marker="o",
    linewidth=2,
    label="5G"
)

ax.set_title(
    "Mobile Network Population Coverage, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Population Coverage (%)")

ax.set_xticks(YEARS)
ax.set_ylim(0, 105)

ax.legend(frameon=False)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_06_LTE_5G_Coverage.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 7. Fixed and Mobile Broadband Traffic

Fixed and mobile broadband traffic are examined over the 2020–2024 period to assess changes in network usage and data consumption. Because the two indicators are measured using the same traffic unit, their trajectories can be compared directly.

In [35]:
# ============================================================
# FIGURE 7 — FIXED AND MOBILE BROADBAND TRAFFIC
# ============================================================

fixed_traffic = get_indicator("i135tfb")
mobile_traffic = get_indicator("i136mwi")

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    fixed_traffic["Year"],
    fixed_traffic["Value"],
    marker="o",
    linewidth=2,
    label="Fixed broadband traffic"
)

ax.plot(
    mobile_traffic["Year"],
    mobile_traffic["Value"],
    marker="o",
    linewidth=2,
    label="Mobile broadband traffic"
)

ax.set_title(
    "Fixed and Mobile Broadband Traffic in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Traffic (Exabytes)")

ax.set_xticks(YEARS)
ax.ticklabel_format(style="plain", axis="y")

ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)

save_figure(
    "Figure_07_Broadband_Traffic.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 8. Evolution of Traditional Communication Services

Fixed outgoing voice traffic and SMS usage are normalized to their respective 2020 levels to permit comparison between indicators measured in different units. An index value of 100 represents the 2020 level of each service.

In [36]:
# ============================================================
# FIGURE 8 — TRADITIONAL COMMUNICATION SERVICES
# Fixed outgoing voice traffic and SMS sent
# Normalized to 2020 = 100
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. DEFINE VERIFIED FIGURE OUTPUT DIRECTORY
# ------------------------------------------------------------

FIGURES_DIR = FIGURE_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Figure will be saved to:")
print(FIGURES_DIR)


# ------------------------------------------------------------
# 2. DEFINE VERIFIED INDICATOR CODES
# ------------------------------------------------------------

voice_code = "i133fo"
sms_code = "i133sms"


# ------------------------------------------------------------
# 3. RETRIEVE INDICATORS
# ------------------------------------------------------------

voice = get_indicator(voice_code).copy()
sms = get_indicator(sms_code).copy()

voice = voice[
    voice["Year"].isin([2020, 2021, 2022, 2023, 2024])
].sort_values("Year")

sms = sms[
    sms["Year"].isin([2020, 2021, 2022, 2023, 2024])
].sort_values("Year")


# ------------------------------------------------------------
# 4. VALIDATE INDICATORS
# ------------------------------------------------------------

assert not voice.empty, \
    "ERROR: Fixed outgoing voice indicator was not found."

assert not sms.empty, \
    "ERROR: SMS indicator was not found."

assert voice["IndicatorCode"].eq("i133fo").all(), \
    "ERROR: Unexpected voice indicator code."

assert sms["IndicatorCode"].eq("i133sms").all(), \
    "ERROR: Unexpected SMS indicator code."

assert voice["Unit"].eq("minutes").all(), \
    "ERROR: Voice unit is not minutes."

assert sms["Unit"].str.lower().eq("messages").all(), \
    "ERROR: SMS unit is not messages."

expected_years = {2020, 2021, 2022, 2023, 2024}

assert set(voice["Year"]) == expected_years, \
    "ERROR: Voice series does not contain exactly 2020–2024."

assert set(sms["Year"]) == expected_years, \
    "ERROR: SMS series does not contain exactly 2020–2024."

assert voice["Value"].notna().all(), \
    "ERROR: Missing values found in voice series."

assert sms["Value"].notna().all(), \
    "ERROR: Missing values found in SMS series."


# ------------------------------------------------------------
# 5. DISPLAY VERIFIED RAW DATA
# ------------------------------------------------------------

print("\nVERIFIED FIXED OUTGOING VOICE TRAFFIC:")
display(
    voice[
        ["IndicatorCode", "Indicator", "Unit", "Year", "Value"]
    ]
)

print("\nVERIFIED SMS SENT:")
display(
    sms[
        ["IndicatorCode", "Indicator", "Unit", "Year", "Value"]
    ]
)


# ------------------------------------------------------------
# 6. NORMALIZE TO 2020 = 100
# ------------------------------------------------------------

voice_base = voice.loc[
    voice["Year"] == 2020,
    "Value"
].iloc[0]

sms_base = sms.loc[
    sms["Year"] == 2020,
    "Value"
].iloc[0]

voice["Index_2020_100"] = (
    voice["Value"] / voice_base
) * 100

sms["Index_2020_100"] = (
    sms["Value"] / sms_base
) * 100


# ------------------------------------------------------------
# 7. DISPLAY NORMALIZED VALUES
# ------------------------------------------------------------

print("\nNORMALIZED FIXED OUTGOING VOICE TRAFFIC:")
display(
    voice[
        ["Year", "Value", "Index_2020_100"]
    ]
)

print("\nNORMALIZED SMS SENT:")
display(
    sms[
        ["Year", "Value", "Index_2020_100"]
    ]
)


# ------------------------------------------------------------
# 8. VERIFY 2020 BASELINES
# ------------------------------------------------------------

assert abs(
    voice.loc[
        voice["Year"] == 2020,
        "Index_2020_100"
    ].iloc[0] - 100
) < 1e-9

assert abs(
    sms.loc[
        sms["Year"] == 2020,
        "Index_2020_100"
    ].iloc[0] - 100
) < 1e-9


# ------------------------------------------------------------
# 9. CREATE FIGURE
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    voice["Year"],
    voice["Index_2020_100"],
    marker="o",
    linewidth=2,
    label="Fixed outgoing voice traffic"
)

ax.plot(
    sms["Year"],
    sms["Index_2020_100"],
    marker="o",
    linewidth=2,
    label="SMS sent"
)

# 2020 baseline
ax.axhline(
    y=100,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)


# ------------------------------------------------------------
# 10. FORMAT FIGURE
# ------------------------------------------------------------

ax.set_title(
    "Traditional Communication Services in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")

ax.set_ylabel(
    "Index (2020 = 100)"
)

ax.set_xticks(
    [2020, 2021, 2022, 2023, 2024]
)

ax.legend(
    frameon=False
)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


# ------------------------------------------------------------
# 11. SAVE FIGURE
# ------------------------------------------------------------

figure_path = (
    FIGURES_DIR
    / "Figure_08_Voice_SMS_Normalized.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)


# ------------------------------------------------------------
# 12. DISPLAY FIGURE
# ------------------------------------------------------------

plt.show()


# ------------------------------------------------------------
# 13. VERIFY SAVED FILE
# ------------------------------------------------------------

if figure_path.exists():

    print("\n✓ Figure 8 created successfully.")
    print("✓ Correct SMS indicator used: i133sms")
    print("✓ Correct voice indicator used: i133fo")
    print("✓ Both series normalized to 2020 = 100")
    print(f"✓ Location: {figure_path}")

else:

    raise FileNotFoundError(
        f"Figure was not saved: {figure_path}"
    )

Figure will be saved to:
C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\04_outputs\Reproduced_Results\National_Analysis\Figures

VERIFIED FIXED OUTGOING VOICE TRAFFIC:


,IndicatorCode,Indicator,Unit,Year,Value
385,i133fo,Total outgoing voice traffic from (originated ...,minutes,2020,"82,641,199.00"
303,i133fo,Total outgoing voice traffic from (originated ...,minutes,2021,"84,784,753.00"
221,i133fo,Total outgoing voice traffic from (originated ...,minutes,2022,"42,610,420.00"
139,i133fo,Total outgoing voice traffic from (originated ...,minutes,2023,"29,683,565.00"
57,i133fo,Total outgoing voice traffic from (originated ...,minutes,2024,"26,557,212.00"



VERIFIED SMS SENT:


,IndicatorCode,Indicator,Unit,Year,Value
391,i133sms,"SMS sent,",Messages,2020,"660,604,925.00"
309,i133sms,"SMS sent,",Messages,2021,"555,701,626.74"
227,i133sms,"SMS sent,",Messages,2022,"553,607,871.00"
145,i133sms,"SMS sent,",Messages,2023,"412,566,297.00"
63,i133sms,"SMS sent,",Messages,2024,"384,719,599.00"



NORMALIZED FIXED OUTGOING VOICE TRAFFIC:


,Year,Value,Index_2020_100
385,2020,"82,641,199.00",100.00
303,2021,"84,784,753.00",102.59
221,2022,"42,610,420.00",51.56
139,2023,"29,683,565.00",35.92
57,2024,"26,557,212.00",32.14



NORMALIZED SMS SENT:


,Year,Value,Index_2020_100
391,2020,"660,604,925.00",100.00
309,2021,"555,701,626.74",84.12
227,2022,"553,607,871.00",83.80
145,2023,"412,566,297.00",62.45
63,2024,"384,719,599.00",58.24



✓ Figure 8 created successfully.
✓ Correct SMS indicator used: i133sms
✓ Correct voice indicator used: i133fo
✓ Both series normalized to 2020 = 100
✓ Location: C:\Users\user\Desktop\Albania_Telecom_Thesis_Research\04_outputs\Reproduced_Results\National_Analysis\Figures\Figure_08_Voice_SMS_Normalized.png


C:\Users\user\AppData\Local\Temp\ipykernel_23488\1229924252.py:249: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 9. Telecommunications Revenue

Total telecommunications revenue and mobile-network revenue are examined over 2020–2024 to assess the financial development of the sector and its mobile component.

In [37]:
# ============================================================
# FIGURE 9 — TELECOM REVENUE
# ============================================================

total_revenue = get_indicator("i75")
mobile_revenue = get_indicator("i741")

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    total_revenue["Year"],
    total_revenue["Value"],
    marker="o",
    linewidth=2,
    label="Total telecommunications revenue"
)

ax.plot(
    mobile_revenue["Year"],
    mobile_revenue["Value"],
    marker="o",
    linewidth=2,
    label="Mobile-network revenue"
)

ax.set_title(
    "Telecommunications Revenue in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Revenue (ALL)")

ax.set_xticks(YEARS)
ax.ticklabel_format(style="plain", axis="y")

ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)

save_figure(
    "Figure_09_Telecom_Revenue.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 10. Telecommunications Investment

Annual telecommunications investment is examined over 2020–2024 to identify changes in sector capital expenditure during the study period.

In [38]:
# ============================================================
# FIGURE 10 — INVESTMENT
# ============================================================

investment = get_indicator("i81")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    investment["Year"],
    investment["Value"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Telecommunications Investment in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Investment (ALL)")

ax.set_xticks(YEARS)
ax.ticklabel_format(style="plain", axis="y")

ax.grid(axis="y", alpha=0.25)

save_figure(
    "Figure_10_Telecom_Investment.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 11. Revenue and Investment Trends

Total telecommunications revenue and investment are normalized to 2020 = 100 to compare their relative trajectories over the study period. The normalization highlights whether financial performance and investment activity evolved in parallel or diverged.

In [39]:
# ============================================================
# FIGURE 11 — REVENUE VS INVESTMENT, 2020 = 100
# ============================================================

financial_codes = {
    "i75": "Total revenue",
    "i81": "Investment"
}

financial_normalized = normalized_table[
    normalized_table["IndicatorCode"].isin(
        financial_codes.keys()
    )
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for code, label in financial_codes.items():

    series = financial_normalized[
        financial_normalized["IndicatorCode"] == code
    ].sort_values("Year")

    ax.plot(
        series["Year"],
        series["Index_2020_100"],
        marker="o",
        linewidth=2,
        label=label
    )

ax.axhline(
    100,
    linewidth=1,
    linestyle="--",
    alpha=0.6
)

ax.set_title(
    "Revenue and Investment Trends, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Index (2020 = 100)")
ax.set_xticks(YEARS)

ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)

save_figure(
    "Figure_11_Revenue_vs_Investment_Normalized.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure 12. Selected Normalized ICT Trends

Four representative indicators are normalized to 2020 = 100 to compare the relative pace of change across fixed broadband adoption, mobile broadband adoption, data consumption, and sector revenue.

The figure compares relative growth rather than absolute magnitudes because the indicators are measured in different units.

In [40]:
# ============================================================
# FIGURE 12 — SELECTED NORMALIZED ICT TRENDS
# Revised version: excludes ≥100 Mbit/s subscriptions because
# its very large relative growth compresses the other series.
# All included indicators are normalized to 2020 = 100.
# ============================================================

selected_normalized_codes = {
    "i4213tfbbb": "Fixed broadband subscriptions",
    "i271mw": "Mobile broadband subscriptions",
    "i135tfb": "Fixed broadband traffic",
    "i75": "Total revenue"
}

# ------------------------------------------------------------
# 1. SELECT THE INDICATORS
# ------------------------------------------------------------

selected_normalized = normalized_table[
    normalized_table["IndicatorCode"].isin(
        selected_normalized_codes.keys()
    )
].copy()


# ------------------------------------------------------------
# 2. VERIFY DATA BEFORE PLOTTING
# ------------------------------------------------------------

print("Indicators included in Figure 12:\n")

for code, label in selected_normalized_codes.items():

    series = selected_normalized[
        selected_normalized["IndicatorCode"] == code
    ].sort_values("Year")

    print(f"{code} — {label}")

    display(
        series[
            ["IndicatorCode", "Year", "Value", "Index_2020_100"]
        ]
    )


# ------------------------------------------------------------
# 3. CREATE FIGURE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6.5))

for code, label in selected_normalized_codes.items():

    series = selected_normalized[
        selected_normalized["IndicatorCode"] == code
    ].sort_values("Year")

    ax.plot(
        series["Year"],
        series["Index_2020_100"],
        marker="o",
        linewidth=2,
        label=label
    )


# ------------------------------------------------------------
# 4. ADD 2020 BASELINE
# ------------------------------------------------------------

ax.axhline(
    y=100,
    linewidth=1,
    linestyle="--",
    alpha=0.6
)


# ------------------------------------------------------------
# 5. FORMAT FIGURE
# ------------------------------------------------------------

ax.set_title(
    "Selected ICT Indicators in Albania, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")

ax.set_ylabel(
    "Index (2020 = 100)"
)

ax.set_xticks(YEARS)

ax.legend(
    frameon=False,
    loc="upper left"
)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


# ------------------------------------------------------------
# 6. SAVE FIGURE
# ------------------------------------------------------------

save_figure(
    "Figure_12_Selected_Normalized_ICT_Trends.png"
)

plt.show()

print(
    "\n✓ Revised Figure 12 created successfully."
)

print(
    "✓ ≥100 Mbit/s subscriptions excluded from this overview."
)

Indicators included in Figure 12:

i4213tfbbb — Fixed broadband subscriptions


,IndicatorCode,Year,Value,Index_2020_100
75,i4213tfbbb,2020,"508,937.00",100.00
76,i4213tfbbb,2021,"559,394.00",109.91
77,i4213tfbbb,2022,"585,285.00",115.00
78,i4213tfbbb,2023,"632,759.00",124.33
79,i4213tfbbb,2024,"694,635.00",136.49


i271mw — Mobile broadband subscriptions


,IndicatorCode,Year,Value,Index_2020_100
40,i271mw,2020,"1,998,579.00",100.00
41,i271mw,2021,"2,056,444.00",102.90
42,i271mw,2022,"2,140,180.00",107.09
43,i271mw,2023,"2,082,525.00",104.20
44,i271mw,2024,"2,091,266.00",104.64


i135tfb — Fixed broadband traffic


,IndicatorCode,Year,Value,Index_2020_100
15,i135tfb,2020,0.61,100.00
16,i135tfb,2021,0.95,156.24
17,i135tfb,2022,1.02,166.63
18,i135tfb,2023,1.26,207.04
19,i135tfb,2024,1.78,292.58


i75 — Total revenue


,IndicatorCode,Year,Value,Index_2020_100
90,i75,2020,"23,681,375,868.00",100.00
91,i75,2021,"26,912,923,899.00",113.65
92,i75,2022,"28,141,435,031.00",118.83
93,i75,2023,"29,517,241,746.00",124.64
94,i75,2024,"32,811,118,052.00",138.55



✓ Revised Figure 12 created successfully.
✓ ≥100 Mbit/s subscriptions excluded from this overview.


C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_23488\1654656192.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Structural Transformation Analysis

In addition to absolute subscription trends, the composition of fixed broadband subscriptions is examined using speed-category shares.

For each year, the share of a speed category is calculated as:

\[
Share_{i,t} =
\frac{Subscriptions_{i,t}}
{\sum_{j=1}^{3}Subscriptions_{j,t}}
\times100
\]

where the denominator represents the combined subscriptions reported across the selected 10–30 Mbit/s, 30–100 Mbit/s, and ≥100 Mbit/s categories.

These shares therefore describe the composition of the three selected speed categories and should not be interpreted as shares of all fixed broadband subscriptions unless the categories collectively exhaust the total.

In [41]:
# ============================================================
# 17. BROADBAND SPEED CATEGORY SHARES
# ============================================================

speed_codes = {
    "i4213_10to30": "10–30 Mbit/s",
    "i4213_30to100": "30–100 Mbit/s",
    "i4213_G100": "≥100 Mbit/s"
}

speed_share_data = master[
    master["IndicatorCode"].isin(speed_codes.keys())
][
    ["IndicatorCode", "Year", "Value"]
].copy()

speed_share_data["Speed_Category"] = (
    speed_share_data["IndicatorCode"]
    .map(speed_codes)
)

speed_share_data["Selected_Category_Total"] = (
    speed_share_data
    .groupby("Year")["Value"]
    .transform("sum")
)

speed_share_data["Share_Percent"] = (
    speed_share_data["Value"]
    / speed_share_data["Selected_Category_Total"]
) * 100

speed_share_table = (
    speed_share_data[
        [
            "Year",
            "Speed_Category",
            "Value",
            "Selected_Category_Total",
            "Share_Percent"
        ]
    ]
    .sort_values(["Year", "Speed_Category"])
    .reset_index(drop=True)
)

speed_share_table

,Year,Speed_Category,Value,Selected_Category_Total,Share_Percent
0,2020,10–30 Mbit/s,"217,974.00","342,856.00",63.58
1,2020,30–100 Mbit/s,"109,476.00","342,856.00",31.93
2,2020,≥100 Mbit/s,"15,406.00","342,856.00",4.49
3,2021,10–30 Mbit/s,"241,804.00","461,878.00",52.35
4,2021,30–100 Mbit/s,"182,418.00","461,878.00",39.49
5,2021,≥100 Mbit/s,"37,656.00","461,878.00",8.15
6,2022,10–30 Mbit/s,"213,146.00","510,246.00",41.77
7,2022,30–100 Mbit/s,"224,859.00","510,246.00",44.07
8,2022,≥100 Mbit/s,"72,241.00","510,246.00",14.16
9,2023,10–30 Mbit/s,"159,989.00","595,903.00",26.85


In [42]:
share_check = (
    speed_share_table
    .groupby("Year")["Share_Percent"]
    .sum()
)

print("Sum of selected speed-category shares by year:")
print(share_check)

assert np.allclose(
    share_check.values,
    100
)

print("\n✓ Speed-category shares sum to 100% in every year.")

Sum of selected speed-category shares by year:
Year
2020   100.00
2021   100.00
2022   100.00
2023   100.00
2024   100.00
Name: Share_Percent, dtype: float64

✓ Speed-category shares sum to 100% in every year.


In [43]:
speed_share_export = speed_share_table.copy()

speed_share_export["Share_Percent"] = (
    speed_share_export["Share_Percent"].round(2)
)

speed_share_export.to_excel(
    TABLE_DIR / "broadband_speed_category_shares.xlsx",
    index=False
)

print(
    "✓ Saved: outputs/broadband_speed_category_shares.xlsx"
)

✓ Saved: outputs/broadband_speed_category_shares.xlsx


### Figure 13. Composition of Selected Broadband Speed Categories

The relative composition of the three selected fixed broadband speed categories is presented to show the structural migration from lower-speed toward higher-speed subscriptions during 2020–2024.

In [44]:
# ============================================================
# FIGURE 13 — BROADBAND SPEED COMPOSITION
# ============================================================

speed_share_pivot = (
    speed_share_table
    .pivot(
        index="Year",
        columns="Speed_Category",
        values="Share_Percent"
    )
)

speed_share_pivot = speed_share_pivot[
    [
        "10–30 Mbit/s",
        "30–100 Mbit/s",
        "≥100 Mbit/s"
    ]
]

fig, ax = plt.subplots(figsize=(10, 6))

speed_share_pivot.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

ax.set_title(
    "Composition of Selected Fixed Broadband Speed Categories, 2020–2024",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Year")
ax.set_ylabel("Share of Selected Categories (%)")

ax.set_ylim(0, 100)

ax.legend(
    title="Speed category",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

ax.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "Figure_13_Broadband_Speed_Composition.png"
)

C:\Users\user\AppData\Local\Temp\ipykernel_23488\43347316.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
